In [1]:
# ==============================
# Travel Recommendation
# Dataset Loading
# ==============================

from pathlib import Path
import pandas as pd

PROJECT_ROOT = Path.cwd()

if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

HOTELS_PATH = (
    PROJECT_ROOT
    / "travel_capstone_dataset"
    / "hotels.csv"
)

print("Dataset path:", HOTELS_PATH)
print("File exists:", HOTELS_PATH.exists())

hotels = pd.read_csv(HOTELS_PATH)

print("Shape:", hotels.shape)
print("Columns:", hotels.columns.tolist())

display(hotels.head())

Dataset path: c:\Users\VINAY\Desktop\Labmentix Projects\Travel_MLops_Major_Project\travel_capstone_dataset\hotels.csv
File exists: True
Shape: (40552, 8)
Columns: ['travelCode', 'userCode', 'name', 'place', 'days', 'price', 'total', 'date']


,travelCode,userCode,name,place,days,price,total,date
0,0,0,Hotel A,Florianopolis (SC),4,313.02,1252.08,09/26/2019
1,2,0,Hotel K,Salvador (BH),2,263.41,526.82,10/10/2019
2,7,0,Hotel K,Salvador (BH),3,263.41,790.23,11/14/2019
3,11,0,Hotel K,Salvador (BH),4,263.41,1053.64,12/12/2019
4,13,0,Hotel A,Florianopolis (SC),1,313.02,313.02,12/26/2019


In [2]:
print("\nDataset information:")
hotels.info()

print("\nMissing values:")
print(hotels.isnull().sum())

print("\nDuplicate rows:", hotels.duplicated().sum())


Dataset information:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 40552 entries, 0 to 40551
Data columns (total 8 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   travelCode  40552 non-null  int64  
 1   userCode    40552 non-null  int64  
 2   name        40552 non-null  object 
 3   place       40552 non-null  object 
 4   days        40552 non-null  int64  
 5   price       40552 non-null  float64
 6   total       40552 non-null  float64
 7   date        40552 non-null  object 
dtypes: float64(2), int64(3), object(3)
memory usage: 2.5+ MB

Missing values:
travelCode    0
userCode      0
name          0
place         0
days          0
price         0
total         0
date          0
dtype: int64

Duplicate rows: 0


In [3]:
for col in hotels.columns:
    print(
        f"\n--- {col} ---"
    )
    print("Unique:", hotels[col].nunique())
    print(hotels[col].dropna().astype(str).head(10).tolist())


--- travelCode ---
Unique: 40552
['0', '2', '7', '11', '13', '15', '22', '29', '32', '33']

--- userCode ---
Unique: 1310
['0', '0', '0', '0', '0', '0', '0', '0', '0', '0']

--- name ---
Unique: 9
['Hotel A', 'Hotel K', 'Hotel K', 'Hotel K', 'Hotel A', 'Hotel BD', 'Hotel Z', 'Hotel AU', 'Hotel AF', 'Hotel K']

--- place ---
Unique: 9
['Florianopolis (SC)', 'Salvador (BH)', 'Salvador (BH)', 'Salvador (BH)', 'Florianopolis (SC)', 'Natal (RN)', 'Aracaju (SE)', 'Recife (PE)', 'Sao Paulo (SP)', 'Salvador (BH)']

--- days ---
Unique: 4
['4', '2', '3', '4', '1', '2', '2', '4', '2', '4']

--- price ---
Unique: 9
['313.02', '263.41', '263.41', '263.41', '313.02', '242.88', '208.04', '312.83', '139.1', '263.41']

--- total ---
Unique: 36
['1252.08', '526.82', '790.23', '1053.64', '313.02', '485.76', '416.08', '1251.32', '278.2', '1053.64']

--- date ---
Unique: 199
['09/26/2019', '10/10/2019', '11/14/2019', '12/12/2019', '12/26/2019', '01/09/2020', '02/27/2020', '04/16/2020', '05/07/2020', '05/

## Step 1 — Clean dates and inspect destinations

In [4]:
# ==============================
# Recommendation Data Preparation
# ==============================

hotels["date"] = pd.to_datetime(
    hotels["date"],
    format="%m/%d/%Y"
)

print("Destinations:")
print(sorted(hotels["place"].unique()))

print("\nHotels:")
print(sorted(hotels["name"].unique()))

print("\nAvailable durations:")
print(sorted(hotels["days"].unique()))

Destinations:
['Aracaju (SE)', 'Brasilia (DF)', 'Campo Grande (MS)', 'Florianopolis (SC)', 'Natal (RN)', 'Recife (PE)', 'Rio de Janeiro (RJ)', 'Salvador (BH)', 'Sao Paulo (SP)']

Hotels:
['Hotel A', 'Hotel AF', 'Hotel AU', 'Hotel BD', 'Hotel BP', 'Hotel BW', 'Hotel CB', 'Hotel K', 'Hotel Z']

Available durations:
[np.int64(1), np.int64(2), np.int64(3), np.int64(4)]


## Step 2 — Create the recommendation function

In [5]:
# ==============================
# Travel Recommendation Engine
# ==============================

def recommend_hotels(
    destination,
    days,
    max_budget=None,
    top_n=5
):
    candidates = hotels[
        hotels["place"].eq(destination) &
        hotels["days"].eq(days)
    ].copy()

    if max_budget is not None:
        candidates = candidates[
            candidates["total"] <= max_budget
        ]

    if candidates.empty:
        return pd.DataFrame()

    # Rank by lowest total trip cost
    recommendations = (
        candidates
        .sort_values(
            by=["total", "price"]
        )
        .drop_duplicates(
            subset=["name"],
            keep="first"
        )
        .head(top_n)
        .copy()
    )

    return recommendations[
        [
            "name",
            "place",
            "days",
            "price",
            "total",
            "date"
        ]
    ]

## Step 3 — Test

In [6]:
recommendations = recommend_hotels(
    destination="Florianopolis (SC)",
    days=4,
    max_budget=1500,
    top_n=5
)

display(recommendations)

,name,place,days,price,total,date
0,Hotel A,Florianopolis (SC),4,313.02,1252.08,2019-09-26


In [7]:
recommendations = recommend_hotels(
    destination="Salvador (BH)",
    days=2,
    max_budget=1000,
    top_n=5
)

display(recommendations)

,name,place,days,price,total,date
1,Hotel K,Salvador (BH),2,263.41,526.82,2019-10-10


## Step 4 — Add a recommendation score

In [8]:
def recommend_hotels(
    destination,
    days,
    max_budget=None,
    top_n=5
):
    candidates = hotels[
        hotels["place"].eq(destination)
    ].copy()

    if max_budget is not None:
        candidates = candidates[
            candidates["total"] <= max_budget
        ]

    if candidates.empty:
        return pd.DataFrame()

    # Exact duration preferred
    candidates["days_difference"] = (
        candidates["days"] - days
    ).abs()

    recommendations = (
        candidates
        .sort_values(
            by=[
                "days_difference",
                "total",
                "price"
            ]
        )
        .drop_duplicates(
            subset=["name"],
            keep="first"
        )
        .head(top_n)
        .copy()
    )

    return recommendations[
        [
            "name",
            "place",
            "days",
            "price",
            "total",
            "date"
        ]
    ]